In [ ]:
from google.colab import drive

drive.mount("/content/drive")
BASE_DIR = "/content/drive/MyDrive/Nexlyra"

Mounted at /content/drive


In [ ]:
import os
import gc
import random
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import StandardScaler

In [ ]:
# DEFINE DATASET CLASS FIRST (Prevents Unpickling Error)
class HydroSpectrogramDataset(Dataset):
    def __init__(self, acoustic_data, physics_data, targets):
        self.acoustic = acoustic_data
        self.physics = physics_data
        self.targets = targets

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        acoust = torch.tensor(self.acoustic[idx], dtype=torch.float32)
        phys = torch.tensor(self.physics[idx], dtype=torch.float32)
        target = torch.tensor(self.targets[idx], dtype=torch.float32)
        return acoust, phys, target

In [ ]:
# TASKS 1 & 2: Hardware & Tensor Initialization
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[*] Target Device: {device}")

def load_day1_artifacts(filepath=BASE_DIR+"/15-10-2026-Ravi/day1_processed_tensors.pt"):
    if os.path.exists(filepath):
        checkpoint = torch.load(filepath, map_location=device,weights_only=False)
        print("[*] Successfully loaded Day 1 PyTorch artifacts.")
        return checkpoint
    else:
        print("[!] Day 1 checkpoint not found. Generating runtime synthetic tensors for demonstration.")
        return None

[*] Target Device: cuda


In [ ]:
# TASKS 3-9: Fluid Mechanics & Spatial Feature Engineering
def calculate_cavitation_index(press_kpa, vel_ms, vapor_pressure_kPa=1.2, rho=1000.0):
    """
    Tasks 4-6: Calculate Cavitation Index (Sigma)
    Sigma = (press_kpa - Vapor_Pressure_kPa) / (0.5 * rho * vel_ms^2 / 1000)
    """
    # Prevent division by zero
    vel_safe = np.maximum(vel_ms, 1e-5)
    dynamic_pressure = 0.5 * rho * (vel_safe ** 2) / 1000.0  # converted to kPa
    sigma = (press_kpa - vapor_pressure_kPa) / dynamic_pressure
    return sigma

In [ ]:
def calculate_spectral_flatness(spectrogram_matrix):
    """
    Task 8: Calculate Spectral Flatness (Geometric Mean / Arithmetic Mean of frequency energy)
    Detects artificial square blocks of noise vs natural harmonic curves.
    """
    flat = np.abs(spectrogram_matrix) + 1e-9
    geom_mean = np.exp(np.mean(np.log(flat), axis=(-1, -2)))
    arith_mean = np.mean(flat, axis=(-1, -2))
    return geom_mean / arith_mean

In [ ]:
def calculate_broadband_ratio(spectrogram_matrix):
    """
    Task 9: Broadband Energy Ratio (energy spanning across all frequency bins vs isolated spikes)
    """
    sum_freq = np.sum(spectrogram_matrix, axis=-1)  # sum across frequency bins
    max_freq = np.max(spectrogram_matrix, axis=-1) + 1e-9
    return np.mean(sum_freq / (max_freq * spectrogram_matrix.shape[-1]), axis=(-1))

In [ ]:
# TASK 22: Custom Spatial Transform (SpecAugment)
class RandomFrequencyMasking(object):
    """
    Task 22: Custom PyTorch spatial transform for domain-specific frequency masking.
    """
    def __init__(self, freq_mask_param=2):
        self.freq_mask_param = freq_mask_param

    def __call__(self, tensor):
        # tensor shape: [1, 10, 10]
        c, f, t = tensor.shape
        f_start = random.randint(0, max(0, f - self.freq_mask_param))
        f_end = f_start + random.randint(1, self.freq_mask_param)
        tensor[:, f_start:f_end, :] = 0.0
        return tensor

In [ ]:
# TASKS 18 & 24: Dual-Input Multi-Modal Dataset & CutMix
class MultiModalHydroDataset(Dataset):
    """
    Task 18: Yields (spectrogram_matrix [1, 10, 10], physics_tabular [6], target)
    """
    def __init__(self, acoustic_list, physics_6d_array, targets, transform=None, augment_minority=False):
        self.acoustic = acoustic_list
        self.physics = physics_6d_array
        self.targets = targets
        self.transform = transform
        self.augment_minority = augment_minority

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        acoust = torch.tensor(self.acoustic[idx], dtype=torch.float32)
        if self.transform and self.augment_minority and self.targets[idx] == 1:
            acoust = self.transform(acoust)

        phys = torch.tensor(self.physics[idx], dtype=torch.float32)
        target = torch.tensor(self.targets[idx], dtype=torch.float32)
        return acoust, phys, target

In [ ]:

def dual_input_cutmix(batch_acoust, batch_phys, batch_targets, alpha=1.0):
    """
    Task 24 & 26: CutMix augmentation adapted for dual-input (Image + Tabular) tensors.
    """
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0

    batch_size = batch_acoust.size(0)
    index = torch.randperm(batch_size).to(batch_acoust.device)

    # CutMix box for 10x10 spectrograms
    _, _, H, W = batch_acoust.shape
    cut_rat = np.sqrt(1.0 - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    batch_acoust[:, :, bby1:bby2, bbx1:bbx2] = batch_acoust[index, :, bby1:bby2, bbx1:bbx2]
    batch_phys = lam * batch_phys + (1 - lam) * batch_phys[index]

    targets_a = batch_targets
    targets_b = batch_targets[index]
    return batch_acoust, batch_phys, targets_a, targets_b, lam


In [ ]:
# MAIN EXECUTION ORCHESTRATOR FOR DAY 2
def main():
    print("[*] Initializing Day 2: Cavitation Mathematics & Spatial Feature Engineering...")

    # Load Day 1 checkpoint or generate mock tensors for standalone verification
    checkpoint = load_day1_artifacts()

 # Generate simulated structured telemetry arrays for Tasks 3-17 demonstration
    N = 1000
    np.random.seed(42)
    mock_press = np.random.uniform(300, 1200, N)
    mock_vel = np.random.uniform(4.0, 14.0, N)
    mock_temp = np.random.uniform(5.0, 25.0, N)
    mock_max_amp = np.random.uniform(1.0, 8.0, N)
    mock_contractors = np.random.choice(['Apex_Hydro_Ops', 'Hydro_Phantom_Turbines', 'Global_Fluid_Dynamics'], N, p=[0.4, 0.3, 0.3])

    # Inject spoofed characteristics for Hydro_Phantom_Turbines
    phantom_mask = (mock_contractors == 'Hydro_Phantom_Turbines')
    mock_press[phantom_mask] = np.random.uniform(900, 1100, phantom_mask.sum())
    mock_vel[phantom_mask] = np.random.uniform(5.0, 7.0, phantom_mask.sum())
    mock_max_amp[phantom_mask] = np.random.uniform(6.0, 9.0, phantom_mask.sum())

    # Task 6: Calculate Cavitation Index (Sigma) and flatten to shape (N,)
    sigma_vals = calculate_cavitation_index(mock_press, mock_vel).flatten()

    # Tasks 8 & 9: Generate mock spectrogram matrices [N, 1, 10, 10] & compute spatial features safely
    mock_specs = np.random.rand(N, 1, 10, 10).astype(np.float32)
    mock_specs[phantom_mask, :, 3:7, 3:7] = 9.9  # Unnatural artificial square blocks

    spectral_flatness = calculate_spectral_flatness(mock_specs).flatten()
    broadband_ratio = calculate_broadband_ratio(mock_specs).flatten()

    # Task 10: Create 3-feature fluid-mechanics tensor of shape [N, 3]
    fluid_features = np.stack([sigma_vals, spectral_flatness, broadband_ratio], axis=-1)

    # Task 15: Flag Probable Acoustic Spoof (High Amplitude + Massive Sigma > 2.0)
    probable_spoofs = (mock_max_amp > 5.0) & (sigma_vals > 2.0)
    print(f"[*] Flagged {probable_spoofs.sum()} records as 'Probable_Acoustic_Spoof'.")

    # Task 17: Concatenate 3 fluid-mechanics features with 3 raw tabular features -> [N, 6]
    raw_tabular = np.stack([mock_press, mock_vel, mock_temp], axis=-1)
    scaler_physics = StandardScaler()
    scaled_fluid = scaler_physics.fit_transform(fluid_features)
    scaled_raw = StandardScaler().fit_transform(raw_tabular)
    physics_6d = np.concatenate([scaled_raw, scaled_fluid], axis=-1)

    # Task 34: Save fitted StandardScaler via joblib
    joblib.dump(scaler_physics, BASE_DIR+"/16-10-2026-Ravi/scaler_fluid_mechanics.joblib")

    # Tasks 30-32: Composite deterministic physics rule evaluation
    # Rule: Sigma > 2.0 AND Max_Amplitude > 5.0 AND Broadband_Ratio < 0.2
    rule_violations = (sigma_vals > 2.0) & (mock_max_amp > 5.0) & (broadband_ratio < 0.2)
    violation_contractors = mock_contractors[rule_violations]
    if len(violation_contractors) > 0:
        phantom_violation_share = (violation_contractors == 'Hydro_Phantom_Turbines').mean() * 100
        print(f"[*] Deterministic Physics Rule: {phantom_violation_share:.1f}% of violations belong to Hydro_Phantom_Turbines.")

    # Tasks 27-29: WeightedRandomSampler setup for severe class imbalance
    targets = np.zeros(N, dtype=np.int64)
    targets[np.random.choice(N, int(N * 0.07), replace=False)] = 1  # ~7% positive class

    class_counts = np.bincount(targets)
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[targets]
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

    # Tasks 18 & 19: Dataset & DataLoader initialization
    dataset = MultiModalHydroDataset(mock_specs, physics_6d, targets, transform=RandomFrequencyMasking(), augment_minority=True)
    loader = DataLoader(
        dataset,
        batch_size=256,
        sampler=sampler,
        num_workers=0
    )

    # Task 29: Verify batch class ratio is roughly 50:50 due to WeightedRandomSampler
    for b_acoust, b_phys, b_target in loader:
        pos_ratio_batch = b_target.float().mean().item()
        print(f"[*] Sampled Batch Class Ratio (Positive): {pos_ratio_batch:.2f}")
        assert 0.4 <= pos_ratio_batch <= 0.6, "Weighted sampler ratio mismatch!"
        break

    # Task 35: Print final input shapes
    print(f"[*] Final Input Shapes -> Spatial Branch: {b_acoust.shape[1:]} | Tabular Branch: [{b_phys.shape[1]}]")

    # Tasks 36 & 37: CUDA memory diagnostics & cleanup
    if torch.cuda.is_available():
        print(torch.cuda.memory_summary())
        torch.cuda.empty_cache()
    gc.collect()

    # Task 38: Save engineered dataset artifacts to disk
    torch.save({
        'physics_6d': physics_6d,
        'targets': targets,
        'sigma_vals': sigma_vals
    }, BASE_DIR+"/16-10-2026-Ravi/day2_engineered_features.pt")

    # Task 39: Ready for Day 3
    print("[*] Day 2 spatial feature engineering complete. Ready for Day 3: Custom Vision Transformer Implementation.")

if __name__ == '__main__':
    main()


[*] Initializing Day 2: Cavitation Mathematics & Spatial Feature Engineering...
[*] Successfully loaded Day 1 PyTorch artifacts.
[*] Flagged 602 records as 'Probable_Acoustic_Spoof'.
[*] Sampled Batch Class Ratio (Positive): 0.50
[*] Final Input Shapes -> Spatial Branch: torch.Size([1, 10, 10]) | Tabular Branch: [6]
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0